# 🌴 Tropical Diseases — Clinical Knowledge Graph with LightRAG (Local Mac & Colab)

This notebook builds a **medical Knowledge Graph** from 2,893 tropical disease case reports using **LightRAG** and **Ollama** (100% offline, private, and free — NO API keys or rate limits!).

### ⚡ 2026 SOTA Model Stack (Optimized for Apple Silicon M1 & Colab):
| Component | Model | Size | Why |
| :--- | :--- | :--- | :--- |
| **LLM** | `qwen2.5:3b` | ~1.9 GB | **#1 SOTA 3B model for structured JSON extraction** — faster and more accurate than older 4B models |
| **Embedding** | `bge-m3` or `nomic-embed-text` | 274–560 MB | **Gold-standard medical embeddings** (dense + sparse retrieval, 8k context) |

### 🚀 Dual Platform Support:
- **On your Mac (DataSpell / VS Code / Jupyter)**: Runs 100% locally on Apple Silicon (M1 Metal GPU) using your `.venv`.
- **On Google Colab**: Runs on free T4 GPU.

---
## Step 1: Verify GPU

In [1]:
import sys, platform

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running on Google Colab")
    !nvidia-smi
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU ready: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"💻 Running locally on Mac: {platform.machine()} ({platform.system()})")
    try:
        import torch
        if torch.backends.mps.is_available():
            print("✅ Apple Silicon Metal (MPS) GPU acceleration is available!")
        else:
            print("ℹ️ CPU mode active")
    except ImportError:
        print("ℹ️ Local Python environment ready (PyTorch not required for Ollama)")

💻 Running locally on Mac: arm64 (Darwin)
ℹ️ Local Python environment ready (PyTorch not required for Ollama)


---
## Step 2: Install Ollama & LightRAG

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install zstd and Ollama on Colab
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
    print("✅ Colab dependencies & Ollama installed!")
else:
    # On Mac, check existing .venv packages
    try:
        try:
            import nest_asyncio
        except ImportError:
            import nest_asyncio2 as nest_asyncio
        import lightrag, ollama, tqdm
        print("✅ All dependencies are already installed in your local .venv!")
    except ImportError as e:
        print(f"Installing missing dependency: {e}")
        !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
        print("✅ Dependencies installed!")

✅ All dependencies are already installed in your local .venv!


---
## Step 3: Start Ollama & Pull Models (~45 seconds)

In [3]:
import os, subprocess, time, urllib.request

# Choose models:
LLM_MODEL = "qwen2.5:3b"            # SOTA 3B extraction model (~1.9 GB, fast & accurate JSON)
EMBEDDING_MODEL = "bge-m3"          # Gold standard for medical RAG (1024 dims). Or use "nomic-embed-text" (768 dims)

# 🔑 CRITICAL SPEED OPTIMIZATION:
# Keep BOTH the LLM and Embedding model loaded in RAM simultaneously!
# This eliminates model swapping/thrashing which previously caused 55-minute delays!
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_NUM_PARALLEL"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# Check if Ollama server is already running
ollama_running = False
try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
    ollama_running = True
    print("✅ Ollama server is already running!")
except Exception:
    pass

if not ollama_running:
    print("Starting Ollama server in background...")
    ollama_bin = "/usr/local/bin/ollama" if os.path.exists("/usr/local/bin/ollama") else "ollama"
    subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=os.environ
    )
    time.sleep(4)
    print("✅ Ollama server started with dual-model memory enabled!")

# Pull models (if not already present)
print(f"\nVerifying models ({LLM_MODEL} and {EMBEDDING_MODEL})...")
!ollama pull {LLM_MODEL}
!ollama pull {EMBEDDING_MODEL}

print("\n--- Installed Models ---")
!ollama list

✅ Ollama server is already running!

Verifying models (qwen2.5:3b and bge-m3)...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 5ee4f07cdb9b: 100% ▕██████████████████▏ 1.9 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling b5c0e5cf74cf: 100% ▕██████████████████▏ 7.4 KB                         
pulling 161ddde4c9cd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifes

---
## Step 4: Mount Google Drive (Recommended)
Saves the Knowledge Graph directly to your Google Drive so you **never lose progress** if Colab disconnects.

Skip this cell if you prefer local-only storage.

In [4]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    USE_DRIVE = True
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            STORAGE_DIR = "/content/drive/MyDrive/tropical_kg_storage"
            print("✅ Google Drive mounted. Storage:", STORAGE_DIR)
        except Exception:
            STORAGE_DIR = "/content/tropical_kg_storage"
            print("Using local Colab storage:", STORAGE_DIR)
    else:
        STORAGE_DIR = "/content/tropical_kg_storage"
else:
    STORAGE_DIR = "./tropical_kg_storage"
    print("✅ Running on Mac! KG will be saved locally to:", os.path.abspath(STORAGE_DIR))

os.makedirs(STORAGE_DIR, exist_ok=True)
print(f"📁 Storage directory: {STORAGE_DIR}")

✅ Running on Mac! KG will be saved locally to: /Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/tropical_kg_storage
📁 Storage directory: ./tropical_kg_storage


---
## Step 5: Locate `cases.csv`
If you uploaded `cases.csv` via the Files panel, it will be at `/content/cases.csv`.
If you put it in Google Drive, it will be at `/content/drive/MyDrive/cases.csv`.

In [5]:
import os, csv

possible_paths = [
    "MultiCaRe_Dataset/Demos/medical_datasets/tropical_infectious_diseases_cohort/cases.csv",
    "../MultiCaRe_Dataset/Demos/medical_datasets/tropical_infectious_diseases_cohort/cases.csv",
    "cases.csv",
    "/content/cases.csv",
    "/content/drive/MyDrive/cases.csv",
    "/content/drive/MyDrive/Datasets/cases.csv",
]

CASES_FILE = None
for p in possible_paths:
    if os.path.exists(p):
        CASES_FILE = p
        break

if CASES_FILE is None:
    raise FileNotFoundError("❌ cases.csv not found!")

size_mb = os.path.getsize(CASES_FILE) / 1024 / 1024
with open(CASES_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    all_cases = list(reader)

print(f"✅ Found: {CASES_FILE} ({size_mb:.1f} MB)")
print(f"   Columns: {list(all_cases[0].keys())}")
print(f"   Total cases: {len(all_cases)}")
print(f"   Sample case: {all_cases[0]['case_id']}")

✅ Found: MultiCaRe_Dataset/Demos/medical_datasets/tropical_infectious_diseases_cohort/cases.csv (11.1 MB)
   Columns: ['case_id', 'pmcid', 'gender', 'age', 'case_text']
   Total cases: 2893
   Sample case: PMC9869191_01


---
## Step 6: Build the Knowledge Graph 🏗️

Configure `MAX_CASES` below:
- `5` → Quick smoke test (~1 min)
- `50` → Solid demo (~8-10 min)
- `500` → Research-grade (~1.5 hrs)
- `None` → All 2,893 cases (~4-6 hrs)

In [6]:
import asyncio
import time
import os
import shutil
from functools import partial
from tqdm.auto import tqdm
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc

try:
    import nest_asyncio
except ImportError:
    import nest_asyncio2 as nest_asyncio

nest_asyncio.apply()

# ========== CONFIGURATION ==========
# LLM_MODEL: "qwen2.5:3b" is SOTA for JSON extraction (faster & more accurate than older 4B models)
LLM_MODEL = "qwen2.5:3b"
# EMBEDDING_MODEL: "bge-m3" (1024-dim, medical gold-standard) or "nomic-embed-text" (768-dim, 274MB)
EMBEDDING_MODEL = "bge-m3"
EMBED_DIM = 1024 if "bge-m3" in EMBEDDING_MODEL else 768

MAX_CASES = 50                      # Set to None for all 2,893 cases
CONCURRENCY = 2                     # Safe parallel workers for Apple M1
# ===================================

# 1. Clean storage for a fresh start
if os.path.exists(STORAGE_DIR):
    shutil.rmtree(STORAGE_DIR)
os.makedirs(STORAGE_DIR, exist_ok=True)
print(f"🧹 Cleaned {STORAGE_DIR} for fresh start!")

# 2. Turn off thinking mode to speed up extraction
os.environ["OLLAMA_LLM_THINK"] = "false"
os.environ["OLLAMA_LLM_NUM_PREDICT"] = "2048"

# 3. Dynamic workspace to prevent duplicate document tracking issues
unique_workspace = f"local_kg_{int(time.time())}"
print(f"Using workspace: {unique_workspace}")
print(f"Using LLM: {LLM_MODEL} | Embedding: {EMBEDDING_MODEL} ({EMBED_DIM} dims)")

# 4. Initialize LightRAG
rag = LightRAG(
    working_dir=STORAGE_DIR,
    workspace=unique_workspace,
    llm_model_func=ollama_model_complete,
    llm_model_name=LLM_MODEL,
    llm_model_max_async=CONCURRENCY,
    embedding_func_max_async=CONCURRENCY,
    default_embedding_timeout=120,
    entity_extraction_use_json=True,
    enable_llm_cache=False,
    llm_model_kwargs={
        "options": {
            "think": False,       # Turn off thinking mode for direct JSON generation
            "num_predict": 2048,  # Output budget for entities
            "num_ctx": 4096,      # Context window for faster processing
        }
    },
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=4096,
        func=partial(ollama_embed.func, embed_model=EMBEDDING_MODEL)
    )
)

print("Initializing LightRAG storages...")
await rag.initialize_storages()
print("✅ Storages initialized\n")

# 5. Select cases to process
cases_to_process = all_cases[:MAX_CASES] if MAX_CASES else all_cases
total = len(cases_to_process)
print(f"Ingesting {total} cases using {LLM_MODEL} + {EMBEDDING_MODEL} with {CONCURRENCY} workers...\n")

# 6. Ingest with progress bar and per-case timing
start_total = time.time()
for i, row in enumerate(tqdm(cases_to_process, desc="Building KG", unit="case")):
    start_case = time.time()

    case_id = row["case_id"]
    age = row.get("age", "N/A")
    gender = row.get("gender", "N/A")
    case_text = f"Case {case_id} (Age: {age}, Gender: {gender}): {row['case_text']}"

    await rag.ainsert(case_text, ids=case_id, file_paths=f"{case_id}.txt")

    elapsed = time.time() - start_case
    tqdm.write(f"  ✓ {case_id} indexed in {elapsed:.1f}s")

total_time = time.time() - start_total
print(f"\n{'='*60}")
print(f"✅ Knowledge Graph built! {total} cases in {total_time/60:.1f} minutes")
print(f"   Avg: {total_time/total:.1f}s per case")
print(f"   Saved to: {STORAGE_DIR}")
print(f"{'='*60}")

/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧹 Cleaned ./tropical_kg_storage for fresh start!
Using workspace: local_kg_1790006484
Using LLM: qwen2.5:3b | Embedding: bge-m3 (1024 dims)


INFO: [local_kg_1790006484] Created new empty graph file: ./tropical_kg_storage/local_kg_1790006484/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1790006484/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1790006484/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1790006484/vdb_chunks.json'} 0 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=2, timeout=240
INFO:  - keyword: None/None, host=None, max_async=2, timeout=240
INFO:  - query: None/None, host=None, max_async=2, timeout=240
INFO:  - vlm: None/None, host=None, max_async=2, timeout=240
INFO: [local_kg_1790006484] Process 20490 KV load full_docs with 0 records
INFO: [local_kg_1790006484

Initializing LightRAG storages...
✅ Storages initialized

Ingesting 50 cases using qwen2.5:3b + bge-m3 with 2 workers...



Building KG:   0%|          | 0/50 [00:00<?, ?case/s]INFO: Processing 1 document(s)
INFO: Parsing (native): PMC9869191_01
INFO: Extracting stage 1/1: PMC9869191_01.txt
INFO: Processing d-id: PMC9869191_01
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: PMC9869191_01
INFO: extract LLM func: 2 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:4174b70a0b7ece423794b3d9785e55b5
INFO:  == LLM cache == saving: default:extract:4a07857c478644b5db5a92a51797663b
INFO: Chunk 1 of 1 extracted 10 Ent + 10 Rel PMC9869191_01-chunk-000
INFO: Merging stage 1/1: PMC9869191_01.txt
INFO: Phase 0: Writing recovery indexes for PMC9869191_01: 15 candidate entities, 10 candidate relations
INFO: Phase 1: Processing 10 entities from PMC9869191_01 (async: 4)
INFO: Phase 2: Processing 10 relations from PMC9869191_01 (async: 4)
INFO: Upserting relation VDB: `36-year-old female->left side chest pain`
INFO: Upser

...

  ✓ PMC9712176_01 indexed in 487.6s


INFO:  == LLM cache == saving: default:extract:e5afb4c00e1911d8624ccd4e9bfa4904
ERROR: Failed to extract entities and relationships: C[1/2]: PMC7308416_01-chunk-000: extract LLM func: Worker execution timeout after 480s
ERROR: Traceback (most recent call last):
  File "/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/.venv/lib/python3.14/site-packages/lightrag/utils.py", line 2393, in wait_func
    result = await future
             ^^^^^^^^^^^^
lightrag.utils.WorkerTimeoutError: Worker execution timeout after 480s

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/.venv/lib/python3.14/site-packages/lightrag/operate.py", line 4457, in _process_with_semaphore
    result = await _process_single_content(chunk)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/.v

  ✓ PMC7308416_01 indexed in 480.2s


INFO:  == LLM cache == saving: default:extract:977cdb7bba721857488c347676cedee3
INFO:  == LLM cache == saving: default:extract:ae44e9587b3cc7ddf6589b10feb3bf58
INFO: Chunk 1 of 1 extracted 8 Ent + 3 Rel PMC4901839_01-chunk-000
INFO: Merging stage 1/1: PMC4901839_01.txt
INFO: Phase 0: Writing recovery indexes for PMC4901839_01: 8 candidate entities, 3 candidate relations
INFO: Phase 1: Processing 8 entities from PMC4901839_01 (async: 4)
INFO: Merged: `CT scan` | 1+2
INFO: Phase 2: Processing 3 relations from PMC4901839_01 (async: 4)
INFO: Upserting relation VDB: `boy->neurosurgeon`
INFO: Upserting relation VDB: `CT scan of the head->boy`
INFO: Upserting relation VDB: `boy->hydrocephalus`
INFO: Phase 2 relation processing completed for PMC4901839_01: edges=3 added_entities=0
INFO: Completed merging: 8 entities, 0 extra entities, 3 relations
INFO: [local_kg_1790006484] entities flush: embedding 8 vectors in 1 batch(es) (batch_num=10)
INFO: [local_kg_1790006484] relationships flush: applie

  ✓ PMC4901839_01 indexed in 209.5s

✅ Knowledge Graph built! 50 cases in 201.9 minutes
   Avg: 242.2s per case
   Saved to: ./tropical_kg_storage


---
## Step 7: Query the Knowledge Graph 🔍

In [ ]:
test_query = "What are the common symptoms, lab abnormalities, and risk factors for Dengue fever and Tuberculosis in tropical patients?"
print(f"Query: {test_query}\n")

response = await rag.aquery(test_query, param=QueryParam(mode="hybrid"))

print("=" * 60)
print("LightRAG Diagnostic Response (Hybrid Mode)")
print("=" * 60)
print(response)

### Try More Queries
Change the query below and re-run to explore the Knowledge Graph.

In [ ]:
# Try different queries!
my_query = "What imaging findings distinguish pulmonary tuberculosis from bacterial pneumonia?"

response = await rag.aquery(my_query, param=QueryParam(mode="hybrid"))
print(response)

---
## Step 8: Download Knowledge Graph to Your Mac
Downloads `tropical_kg_storage.zip` to your computer.
Unzip it into your local `Datasets/tropical_kg_storage/` folder to use offline!

In [ ]:
import os, shutil, sys

zip_path = "./tropical_kg_storage"
shutil.make_archive(zip_path, 'zip', STORAGE_DIR)
print(f"✅ Knowledge Graph archived to: {zip_path}.zip")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("Downloading to browser...")